In [0]:
%pip install kafka-python
dbutils.library.restartPython()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS streaming_bronze;

In [0]:
#-----Atualizar aqui quando reiniciar o ngrok-----
kafka_bootstrap = "0.tcp.sa.ngrok.io:28274"

topico = "stocks-kafka"
streaming_path = "workspace.stocks.streaming_bronze"
checkpoint_path = "dbfs:/FileStore/stocks/checkpoints/bronze_stream"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

ticks_schema = StructType([
    StructField("id", StringType(), nullable=True),
    StructField("price", DoubleType(), nullable=True),
    StructField("time", StringType(), nullable=True),
    StructField("exchange", StringType(), nullable=True),
    StructField("quote_type", StringType(), nullable=True),
    StructField("market_hours", StringType(), nullable=True),
    StructField("change_percent", DoubleType(), nullable=True),
    StructField("day_volume", StringType(), nullable=True),
    StructField("change", DoubleType(), nullable=True),
    StructField("last_size", StringType(), nullable=True),
    StructField("price_hint", StringType(), nullable=True),
])

print("-----Schema definido-----")
print(f"Campos: {ticks_schema.fieldNames()}")

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp

kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", topico)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

In [0]:
ticks = (
    kafka_stream
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), ticks_schema).alias("tick"))
    .select("tick.*")
)